# 04 — Evaluación y Tuning
## LoanRisk-ML

Optimizamos los hiperparámetros de LightGBM y XGBoost con Optuna,
seleccionamos el modelo ganador y encontramos el threshold óptimo.

### Objetivos
- Tunear LightGBM y XGBoost con Optuna maximizando AUC-PR en validación
- Seleccionar el modelo ganador por mejor balance AUC-PR y gap train-val
- Encontrar el threshold óptimo maximizando F1
- Construir el scorecard 300-850
- Guardar resultados finales en JSON

### Input
`data/processed/X_train.parquet`, `X_val.parquet`, `X_test.parquet`
`data/processed/y_train.parquet`, `y_val.parquet`, `y_test.parquet`

### Output
`models/LightGBM_best.joblib`, `models/XGBoost_best.joblib`
`models/resultados_finales.json`

## 1. Importaciones

In [1]:
import warnings
warnings.filterwarnings('ignore')

import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import mlflow
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
import xgboost as xgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    f1_score
)

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


## 2. Carga de datos y configuración de rutas

In [2]:
ROOT           = Path('..').resolve()
DATA_PROCESSED = ROOT / 'data' / 'processed'
MODELS_DIR     = ROOT / 'models'
MLFLOW_DIR     = ROOT / 'mlruns'

# Cargar conjuntos procesados
X_train = pd.read_parquet(DATA_PROCESSED / 'X_train.parquet')
X_val   = pd.read_parquet(DATA_PROCESSED / 'X_val.parquet')
X_test  = pd.read_parquet(DATA_PROCESSED / 'X_test.parquet')

y_train = pd.read_parquet(DATA_PROCESSED / 'y_train.parquet').squeeze()
y_val   = pd.read_parquet(DATA_PROCESSED / 'y_val.parquet').squeeze()
y_test  = pd.read_parquet(DATA_PROCESSED / 'y_test.parquet').squeeze()

print(f"X_train: {X_train.shape} — default rate: {y_train.mean():.2%}")
print(f"X_val:   {X_val.shape}   — default rate: {y_val.mean():.2%}")
print(f"X_test:  {X_test.shape}  — default rate: {y_test.mean():.2%}")

X_train: (273814, 76) — default rate: 20.15%
X_val:   (58675, 76)   — default rate: 20.15%
X_test:  (58675, 76)  — default rate: 20.15%


## 3. Configuración de MLflow

In [3]:
mlflow.set_tracking_uri(f"file:///{MLFLOW_DIR}")
mlflow.set_experiment("loanrisk-optuna")

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experimento: loanrisk-optuna")

Tracking URI: file:///C:\Users\micke\OneDrive\Desktop\projects\loanrisk-ml\mlruns
Experimento: loanrisk-optuna


## 4. Funciones objetivo de Optuna

Definimos una función objetivo por modelo que Optuna optimiza.
Cada trial sugiere hiperparámetros, entrena el modelo
y devuelve AUC-PR en validación como métrica a maximizar.

In [4]:
# Calcular scale_pos_weight — ratio negativos/positivos
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.4f}")
print(f"  Negativos (no default): {(y_train == 0).sum():,}")
print(f"  Positivos (default):    {(y_train == 1).sum():,}")

def objective_lgbm(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 1.0),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'scale_pos_weight':  scale_pos_weight,
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': -1
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    p_val = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, p_val)


def objective_xgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.0, 1.0),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'logloss',
        'verbosity': 0
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    p_val = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, p_val)


print("\nFunciones objetivo definidas correctamente")

scale_pos_weight: 3.9625
  Negativos (no default): 218,637
  Positivos (default):    55,177

Funciones objetivo definidas correctamente


## 5. Ejecución de estudios de Optuna

Optuna busca los mejores hiperparámetros para cada modelo
ejecutando múltiples trials de forma inteligente.
Usamos 50 trials por modelo — balance entre calidad y tiempo.

In [5]:
import time

configs = [
    ("LightGBM", objective_lgbm),
    ("XGBoost",  objective_xgb)
]

studies = {}

for name, objective in configs:
    print(f"Optimizando {name} — 50 trials...")
    t0 = time.time()

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    tiempo = time.time() - t0
    studies[name] = study

    print(f"  Mejor AUC-PR Val: {study.best_value:.4f}")
    print(f"  Tiempo total:     {tiempo/60:.1f} min")
    print()

print("Optimización completada.")

Optimizando LightGBM — 50 trials...


  0%|          | 0/50 [00:00<?, ?it/s]

  Mejor AUC-PR Val: 0.5644
  Tiempo total:     12.4 min

Optimizando XGBoost — 50 trials...


  0%|          | 0/50 [00:00<?, ?it/s]

  Mejor AUC-PR Val: 0.5650
  Tiempo total:     10.0 min

Optimización completada.


## 6. Guardar mejores hiperparámetros

In [6]:
best_params = {
    "LightGBM": studies["LightGBM"].best_params,
    "XGBoost":  studies["XGBoost"].best_params
}

with open(MODELS_DIR / "best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

print("Mejores hiperparámetros encontrados:")
for name, params in best_params.items():
    print(f"\n{name}:")
    for k, v in params.items():
        print(f"  {k}: {v}")

Mejores hiperparámetros encontrados:

LightGBM:
  n_estimators: 686
  learning_rate: 0.025103701639564595
  num_leaves: 76
  min_child_samples: 155
  reg_alpha: 0.7878076740015398
  reg_lambda: 0.9951938423814498
  subsample: 0.8848452308849397
  colsample_bytree: 0.5545190284151778

XGBoost:
  n_estimators: 764
  learning_rate: 0.05187689686689319
  max_depth: 5
  min_child_weight: 7
  reg_alpha: 0.7674983988271411
  reg_lambda: 0.97270743431834
  subsample: 0.8162137840181485
  colsample_bytree: 0.7983250702689573


## 7. Reentrenamiento con mejores hiperparámetros

Entrenamos cada modelo con los hiperparámetros óptimos encontrados por Optuna
y los registramos en MLflow.

In [7]:
import time

resultados_tuning = []

for name, study in studies.items():
    print(f"Reentrenando {name} con mejores hiperparámetros...")
    t0 = time.time()

    params = study.best_params.copy()
    params['random_state'] = 42
    params['n_jobs'] = -1
    params['scale_pos_weight'] = scale_pos_weight

    if name == "LightGBM":
        params['verbosity'] = -1
        model = lgb.LGBMClassifier(**params)
    else:
        params['verbosity'] = 0
        params['eval_metric'] = 'logloss'
        model = xgb.XGBClassifier(**params)

    model.fit(X_train, y_train)
    tiempo = time.time() - t0

    p_train = model.predict_proba(X_train)[:, 1]
    p_val   = model.predict_proba(X_val)[:, 1]
    p_test  = model.predict_proba(X_test)[:, 1]

    auc_pr_train  = average_precision_score(y_train, p_train)
    auc_pr_val    = average_precision_score(y_val,   p_val)
    auc_pr_test   = average_precision_score(y_test,  p_test)
    auc_roc_val   = roc_auc_score(y_val, p_val)
    gap_train_val = auc_pr_train - auc_pr_val

    fpr, tpr, _ = roc_curve(y_val, p_val)
    ks   = (tpr - fpr).max()
    gini = 2 * auc_roc_val - 1

    with mlflow.start_run(run_name=f"{name}_tuned"):
        mlflow.log_params(params)
        mlflow.log_metrics({
            'auc_pr_train':  round(auc_pr_train, 4),
            'auc_pr_val':    round(auc_pr_val, 4),
            'auc_pr_test':   round(auc_pr_test, 4),
            'auc_roc_val':   round(auc_roc_val, 4),
            'gap_train_val': round(gap_train_val, 4),
            'ks':            round(ks, 4),
            'gini':          round(gini, 4),
        })
        mlflow.sklearn.log_model(model, name)

    joblib.dump(model, MODELS_DIR / f"{name}_best.joblib")

    resultados_tuning.append({
        'Modelo':        name,
        'AUC-PR Train':  round(auc_pr_train, 4),
        'AUC-PR Val':    round(auc_pr_val, 4),
        'AUC-PR Test':   round(auc_pr_test, 4),
        'AUC-ROC Val':   round(auc_roc_val, 4),
        'KS':            round(ks, 4),
        'Gini':          round(gini, 4),
        'Gap Train-Val': round(gap_train_val, 4),
        'Tiempo (s)':    round(tiempo, 2)
    })

    print(f"  AUC-PR Train: {auc_pr_train:.4f}")
    print(f"  AUC-PR Val:   {auc_pr_val:.4f}")
    print(f"  AUC-PR Test:  {auc_pr_test:.4f}")
    print(f"  AUC-ROC Val:  {auc_roc_val:.4f}")
    print(f"  KS:           {ks:.4f}")
    print(f"  Gini:         {gini:.4f}")
    print(f"  Gap:          {gap_train_val:.4f}")
    print(f"  Tiempo:       {tiempo:.1f}s")
    print()

print("Reentrenamiento completado.")

Reentrenando LightGBM con mejores hiperparámetros...


2026/04/06 16:58:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/06 16:58:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  AUC-PR Train: 0.6455
  AUC-PR Val:   0.5644
  AUC-PR Test:  0.5703
  AUC-ROC Val:  0.7868
  KS:           0.4159
  Gini:         0.5737
  Gap:          0.0811
  Tiempo:       12.4s

Reentrenando XGBoost con mejores hiperparámetros...


2026/04/06 16:59:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/06 16:59:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  AUC-PR Train: 0.6239
  AUC-PR Val:   0.5650
  AUC-PR Test:  0.5700
  AUC-ROC Val:  0.7867
  KS:           0.4169
  Gini:         0.5734
  Gap:          0.0589
  Tiempo:       13.3s

Reentrenamiento completado.


## 8. Comparación de modelos tuneados y selección del ganador

Comparamos AUC-PR en val y test, y el gap train-val para detectar overfitting.
El modelo ganador es el que mejor generaliza — no necesariamente el de mayor AUC-PR en val.

In [8]:
df_resumen = pd.DataFrame(resultados_tuning)
df_resumen = df_resumen.sort_values('AUC-PR Val', ascending=False).reset_index(drop=True)

print("=" * 80)
print("COMPARACIÓN DE MODELOS TUNEADOS")
print("=" * 80)
print(df_resumen.to_string(index=False))
print("=" * 80)

# Selección del modelo ganador
# Criterio: mejor AUC-PR Val con menor gap Train-Val
mejor = df_resumen.sort_values(['AUC-PR Val', 'Gap Train-Val'],
                                ascending=[False, True]).iloc[0]

nombre_ganador = mejor['Modelo']
print(f"\nModelo ganador: {nombre_ganador}")
print(f"  AUC-PR Val:    {mejor['AUC-PR Val']}")
print(f"  AUC-PR Test:   {mejor['AUC-PR Test']}")
print(f"  Gap Train-Val: {mejor['Gap Train-Val']}")

COMPARACIÓN DE MODELOS TUNEADOS
  Modelo  AUC-PR Train  AUC-PR Val  AUC-PR Test  AUC-ROC Val     KS   Gini  Gap Train-Val  Tiempo (s)
 XGBoost        0.6239      0.5650       0.5700       0.7867 0.4169 0.5734         0.0589       13.35
LightGBM        0.6455      0.5644       0.5703       0.7868 0.4159 0.5737         0.0811       12.39

Modelo ganador: XGBoost
  AUC-PR Val:    0.565
  AUC-PR Test:   0.57
  Gap Train-Val: 0.0589


## 9. Carga del modelo ganador y optimización del threshold

Buscamos el threshold que maximiza F1 en el conjunto de test.
El threshold por defecto de 0.5 no es óptimo en datasets desbalanceados.

In [9]:
from sklearn.metrics import precision_score, recall_score

# Cargar modelo ganador
model = joblib.load(MODELS_DIR / f"{nombre_ganador}_best.joblib")

# Predicciones en test
proba_test = model.predict_proba(X_test)[:, 1]

# Calcular F1 para cada threshold
precisions, recalls, thresholds = precision_recall_curve(y_test, proba_test)
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)

# Threshold óptimo
idx_optimo = f1_scores.argmax()
threshold  = thresholds[idx_optimo]
f1_optimo  = f1_scores[idx_optimo]

print(f"Threshold óptimo: {threshold:.4f}")
print(f"F1 en threshold óptimo: {f1_optimo:.4f}")
print()
print("Comparación de thresholds:")
for t in [0.30, 0.40, threshold, 0.50]:
    pred = (proba_test >= t).astype(int)
    f1   = f1_score(y_test, pred)
    prec = precision_score(y_test, pred)
    rec  = recall_score(y_test, pred)
    print(f"  Threshold {t:.4f} — F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")

Threshold óptimo: 0.5387
F1 en threshold óptimo: 0.5076

Comparación de thresholds:
  Threshold 0.3000 — F1: 0.4289 | Precision: 0.2821 | Recall: 0.8942
  Threshold 0.4000 — F1: 0.4667 | Precision: 0.3313 | Recall: 0.7894
  Threshold 0.5387 — F1: 0.5076 | Precision: 0.4351 | Recall: 0.6093
  Threshold 0.5000 — F1: 0.4997 | Precision: 0.4009 | Recall: 0.6633


## 10. Construcción del Scorecard 300-850

Transformamos la probabilidad de default en un score entendible
por el negocio — similar al score FICO usado en la industria.

In [10]:
def prob_a_score(prob, score_min=300, score_max=850):
    """
    Convierte probabilidad de default a score crediticio.
    Mayor probabilidad de default → menor score.
    """
    return int(score_max - (score_max - score_min) * prob)

# Aplicar scorecard al conjunto de test
scores_test = np.array([prob_a_score(p) for p in proba_test])

# Distribución del score por clase
scores_no_default = scores_test[y_test == 0]
scores_default    = scores_test[y_test == 1]

print("Distribución del score por clase:")
print(f"  No default — Media: {scores_no_default.mean():.0f} | Min: {scores_no_default.min()} | Max: {scores_no_default.max()}")
print(f"  Default    — Media: {scores_default.mean():.0f} | Min: {scores_default.min()} | Max: {scores_default.max()}")

print("\nEjemplos de conversión:")
for prob in [0.05, 0.10, 0.20, 0.35, 0.50, 0.70, 0.90]:
    score = prob_a_score(prob)
    riesgo = "Bajo" if score >= 700 else "Medio" if score >= 600 else "Alto"
    print(f"  Prob default: {prob:.0%} → Score: {score} → Riesgo: {riesgo}")

Distribución del score por clase:
  No default — Media: 651 | Min: 300 | Max: 846
  Default    — Media: 511 | Min: 300 | Max: 844

Ejemplos de conversión:
  Prob default: 5% → Score: 822 → Riesgo: Bajo
  Prob default: 10% → Score: 795 → Riesgo: Bajo
  Prob default: 20% → Score: 740 → Riesgo: Bajo
  Prob default: 35% → Score: 657 → Riesgo: Medio
  Prob default: 50% → Score: 575 → Riesgo: Alto
  Prob default: 70% → Score: 465 → Riesgo: Alto
  Prob default: 90% → Score: 355 → Riesgo: Alto


## 11. Guardar resultados finales

Guardamos el modelo ganador, threshold óptimo y métricas en JSON
para uso en producción y en la API de FastAPI.

In [11]:
resultados_finales = {
    "modelo":           nombre_ganador,
    "threshold":        round(float(threshold), 4),
    "metricas": {
        "auc_pr_val":    round(float(average_precision_score(y_val, model.predict_proba(X_val)[:, 1])), 4),
        "auc_pr_test":   round(float(average_precision_score(y_test, proba_test)), 4),
        "auc_roc_val":   round(float(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])), 4),
        "ks":            round(float(mejor['KS']), 4),
        "gini":          round(float(mejor['Gini']), 4),
        "f1_optimo":     round(float(f1_optimo), 4),
        "gap_train_val": round(float(mejor['Gap Train-Val']), 4)
    },
    "scorecard": {
        "score_min": 300,
        "score_max": 850,
        "score_medio_no_default": int(scores_no_default.mean()),
        "score_medio_default":    int(scores_default.mean())
    },
    "escala_riesgo": {
        "bajo":  "score >= 700",
        "medio": "score >= 600 y < 700",
        "alto":  "score < 600"
    }
}

with open(MODELS_DIR / 'resultados_finales.json', 'w') as f:
    json.dump(resultados_finales, f, indent=2)

print("Resultados finales guardados en models/resultados_finales.json")
print()
print(json.dumps(resultados_finales, indent=2))

Resultados finales guardados en models/resultados_finales.json

{
  "modelo": "XGBoost",
  "threshold": 0.5387,
  "metricas": {
    "auc_pr_val": 0.565,
    "auc_pr_test": 0.57,
    "auc_roc_val": 0.7867,
    "ks": 0.4169,
    "gini": 0.5734,
    "f1_optimo": 0.5076,
    "gap_train_val": 0.0589
  },
  "scorecard": {
    "score_min": 300,
    "score_max": 850,
    "score_medio_no_default": 650,
    "score_medio_default": 511
  },
  "escala_riesgo": {
    "bajo": "score >= 700",
    "medio": "score >= 600 y < 700",
    "alto": "score < 600"
  }
}
